# 6. 模型格式与序列化

高效的模型存储格式直接影响加载速度、内存占用和跨平台兼容性。在产业级部署中，模型从训练到推理往往需要经历多次格式转换，理解每种格式的设计哲学和适用场景是端侧部署工程师的核心能力。

## 什么是模型格式？

模型格式是模型权重、结构和元数据的序列化方式。核心格式包括：
- **PyTorch (.pt/.pth)**：Pickle序列化，灵活但不安全，加载慢
- **ONNX (.onnx)**：跨框架标准格式，Protobuf序列化，支持算子标准化
- **SafeTensors**：HuggingFace推出的安全格式，零拷贝mmap加载，无Pickle安全风险
- **GGUF**：llama.cpp使用的格式，支持多种量化类型，mmap零拷贝加载
- **TensorRT (.engine/.plan)**：NVIDIA推理引擎的优化格式，层融合+内核自动调优
- **TFLite (.tflite)**：Google端侧推理格式，FlatBuffers序列化，支持Android/嵌入式
- **Core ML (.mlpackage/.mlmodelc)**：Apple端侧格式，ANE优化，支持iOS/macOS

## 为什么模型格式重要？

1. **加载速度**：mmap零拷贝加载（SafeTensors/GGUF）比Pickle反序列化快10-100倍
2. **安全性**：Pickle反序列化可执行任意代码，SafeTensors避免了此风险
3. **内存效率**：量化格式（GGUF）直接加载低精度权重，无需先加载FP16再量化
4. **跨平台**：ONNX等标准格式使得模型可在不同框架和硬件间迁移
5. **推理性能**：TensorRT等引擎格式通过层融合和内核选择实现最优推理性能
6. **分发效率**：大模型（70B+）需要分片存储，格式设计直接影响下载和加载体验

## 核心原理

**mmap零拷贝**：操作系统将文件直接映射到进程地址空间，无需将数据从内核空间拷贝到用户空间。首次访问时触发缺页中断（page fault），操作系统按需加载对应页面：
$$t_{\text{mmap}} = O(1) \quad \text{（映射本身）}, \quad t_{\text{access}} = O(\text{实际访问的页面数})$$
vs 传统读取：$t_{\text{read}} = O(\text{file size})$（必须全部读入）

mmap的关键优势：
- **惰性加载**：只加载实际使用的权重，多LoRA场景下可按需加载不同适配器
- **跨进程共享**：多个进程可共享同一mmap映射，减少总内存占用
- **大模型支持**：70B模型无需全部加载到内存，按层按需访问

**Block量化（GGUF）**：将权重按block分组量化，每个block独立的scale和offset：
$$W_{q,k} = \text{round}(W_k / s_k), \quad s_k = \frac{\max(|W_k|)}{q_{\max}}$$

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import json
import struct
import os
import time

torch.manual_seed(42)
print(f"PyTorch version: {torch.__version__}")

### GGUF格式详解

#### 什么是GGUF？

GGUF（GPT-Generated Unified Format）是llama.cpp的标准格式，支持元数据嵌入、多种量化类型（Q2_K到Q8_0）和mmap零拷贝加载。GGUF是当前端侧CPU推理（尤其是Apple Silicon和x86 AVX）最主流的格式。

#### 为什么GGUF有效？

1. **mmap零拷贝加载**：操作系统将文件直接映射到进程地址空间，加载时间$O(1)$
2. **K-Quant混合量化**：对权重矩阵的不同部分使用不同精度，精度-压缩比更优
3. **自包含格式**：词表、超参数、量化参数全部嵌入文件，单文件即可推理
4. **对齐要求**：数据按32字节对齐，适配AVX/SSE向量指令

#### GGUF文件结构

```
| Magic (4B) | Version (4B) | Tensor Count (8B) | Metadata KV Count (8B) |
| Metadata KV Pairs ... |
| Tensor Info (name + dims + type + offset) ... |
| Padding to Alignment |
| Tensor Data ... |
```

#### GGUF的Block量化原理

将权重按block分组量化：
$$W_{q,k} = \text{round}(W_k / s_k), \quad s_k = \frac{\max(|W_k|)}{q_{\max}}$$

K-Quant分层结构：
- super-block（256权重）：存储全局scale $s_{\text{super}}$
- sub-block（32权重）：存储局部scale $s_{\text{sub}}$和offset
- 最终量化值：$W_{q,i} = \text{round}(W_i / (s_{\text{super}} \cdot s_{\text{sub}}))$

K-Quant类型对比（7B模型典型指标）：
| 类型 | bits/weight | 文件大小 | Wiki2 PPL | 适用场景 |
|------|------------|---------|-----------|----------|
| Q8_0 | 8.5 | ~7.4GB | ~5.40 | 高精度端侧 |
| Q6_K | 6.59 | ~5.7GB | ~5.45 | 精度-大小均衡 |
| Q5_K_M | 5.69 | ~4.9GB | ~5.52 | 推荐默认选择 |
| Q4_K_M | 4.80 | ~4.1GB | ~5.60 | 端侧8GB内存 |
| Q3_K_M | 3.91 | ~3.4GB | ~5.82 | 端侧4GB内存 |
| Q2_K | 2.56 | ~2.2GB | ~6.45 | 极限压缩 |

In [ ]:
class GGUFFileSimulator:
    """GGUF文件格式模拟"""
    GGUF_MAGIC = 0x47475546  # 'GGUF' in little-endian
    TYPES = {0: 'F32', 1: 'F16', 2: 'Q4_0', 3: 'Q4_1', 6: 'Q5_0', 7: 'Q5_1',
             8: 'Q8_0', 12: 'Q4_K', 13: 'Q5_K', 14: 'Q6_K'}

    BYTES_PER_ELEMENT = {
        'F32': 4.0, 'F16': 2.0,
        'Q4_0': 18/32, 'Q4_1': 20/32,
        'Q5_0': 22/32, 'Q5_1': 24/32,
        'Q8_0': 34/32,
        'Q4_K': 144/256, 'Q5_K': 176/256, 'Q6_K': 210/256,
    }

    def __init__(self):
        self.metadata = {}
        self.tensor_infos = {}
        self.alignment = 32

    def add_metadata(self, key, value):
        self.metadata[key] = value

    def add_tensor_info(self, name, shape, dtype='Q4_K', offset=0):
        n_elements = 1
        for s in shape:
            n_elements *= s
        bpe = self.BYTES_PER_ELEMENT.get(dtype, 2)
        size_bytes = int(n_elements * bpe)
        self.tensor_infos[name] = {
            'shape': shape, 'dtype': dtype, 'offset': offset, 'size': size_bytes,
            'bits_per_weight': bpe * 8
        }

    def estimate_file_size(self):
        header_size = 64
        metadata_size = sum(len(k) + 16 for k in self.metadata)
        tensor_info_size = sum(len(k) + 64 for k in self.tensor_infos)
        tensor_data_size = sum(t['size'] for t in self.tensor_infos.values())
        return header_size + metadata_size + tensor_info_size + tensor_data_size

gguf = GGUFFileSimulator()
gguf.add_metadata('general.architecture', 'llama')
gguf.add_metadata('llm.context_length', 4096)
gguf.add_metadata('llm.embedding_length', 4096)
gguf.add_metadata('llm.layer_count', 32)
gguf.add_metadata('llm.attention.head_count', 32)

gguf.add_tensor_info('token_embd.weight', [4096, 32000], 'Q4_K')
for i in range(32):
    gguf.add_tensor_info(f'blk.{i}.attn_q.weight', [4096, 4096], 'Q4_K')
    gguf.add_tensor_info(f'blk.{i}.attn_k.weight', [4096, 512], 'Q4_K')
    gguf.add_tensor_info(f'blk.{i}.attn_v.weight', [4096, 512], 'Q4_K')
    gguf.add_tensor_info(f'blk.{i}.attn_output.weight', [4096, 4096], 'Q4_K')
    gguf.add_tensor_info(f'blk.{i}.ffn_gate.weight', [4096, 11008], 'Q4_K')
    gguf.add_tensor_info(f'blk.{i}.ffn_up.weight', [4096, 11008], 'Q4_K')
    gguf.add_tensor_info(f'blk.{i}.ffn_down.weight', [11008, 4096], 'Q4_K')

print(f"=== GGUF格式模拟 (7B模型) ===")
print(f"元数据条目: {len(gguf.metadata)}")
print(f"张量条目: {len(gguf.tensor_infos)}")
print(f"\n元数据内容:")
for k, v in gguf.metadata.items():
    print(f"  {k}: {v}")

print(f"\n量化类型 bits/weight 对比:")
for dtype_name, bpe in sorted(GGUFFileSimulator.BYTES_PER_ELEMENT.items(), key=lambda x: x[1]):
    print(f"  {dtype_name:<8} {bpe*8:.2f} bits/weight")

est_size = gguf.estimate_file_size()
print(f"\nQ4_K估算文件大小: {est_size/1024/1024/1024:.2f} GB")
print(f"FP16原始大小: {7.0:.2f} GB")
print(f"压缩比: {7.0/est_size*1024*1024*1024:.1f}x")

print(f"\nGGUF核心特性:")
print(f"1. 单文件格式: 元数据+权重一体")
print(f"2. mmap支持: 零拷贝加载，多进程共享")
print(f"3. 丰富量化: Q4_0到Q6_K多种选择")
print(f"4. 扩展性: 键值对元数据，易于扩展")
print(f"5. 对齐: 32字节对齐，适配SIMD指令")

### SafeTensors格式

#### 什么是SafeTensors？

HuggingFace推出的安全模型存储格式，使用零拷贝mmap加载，无Pickle反序列化安全风险。已成为HuggingFace Hub的默认模型格式。

#### 为什么用SafeTensors？

1. **安全性**：Pickle反序列化可执行任意代码（`__reduce__`方法），SafeTensors使用纯数据格式，无代码执行风险
2. **加载速度**：mmap零拷贝加载比Pickle反序列化快10-100倍
3. **跨语言**：简单的二进制格式，可被任何编程语言读取（Rust/Go/JS/C++）
4. **惰性加载**：可只加载需要的张量，支持大模型按需加载
5. **分片支持**：大模型可拆分为多个分片文件（model-00001-of-00008.safetensors）

#### SafeTensors的文件结构

```
| Header Size (8 bytes, LE uint64) | JSON Header | Padding | Tensor Data |
```

- Header Size：头部大小，8字节无符号小端整数
- JSON Header：张量元数据（名称、数据类型、形状、偏移量），偏移量相对于数据区起始
- Padding：数据区按8字节对齐
- Tensor Data：连续存储的张量数据，每个张量按其dtype的自然对齐排列

加载时间对比：
- Pickle：$t = O(\text{file size})$（需反序列化+拷贝）
- SafeTensors：$t = O(1)$（mmap映射，按需加载）

#### PyTorch Pickle安全风险

```python
# 恶意.pt文件可以执行任意代码
import pickle
class Exploit:
    def __reduce__(self):
        return (os.system, ('rm -rf /',))
# torch.load('malicious.pt') 会执行上述命令
```
这就是为什么HuggingFace强制使用SafeTensors，且`torch.load`从PyTorch 2.0起默认`weights_only=True`。

In [ ]:
class SafeTensorsSimulator:
    """SafeTensors格式模拟"""
    DTYPE_MAP = {'F16': 2, 'F32': 4, 'BF16': 2, 'F64': 8, 'BOOL': 1, 'U8': 1, 'I8': 1, 'I32': 4, 'I64': 8}

    def __init__(self):
        self.tensors = {}

    def add_tensor(self, name, shape, dtype='F16'):
        self.tensors[name] = {'shape': shape, 'dtype': dtype}

    def simulate_save(self):
        header = {}
        offset = 0
        for name, info in self.tensors.items():
            n_elements = 1
            for s in info['shape']:
                n_elements *= s
            dtype_size = self.DTYPE_MAP.get(info['dtype'], 2)
            size = n_elements * dtype_size
            header[name] = {
                'dtype': info['dtype'],
                'shape': info['shape'],
                'data_offsets': [offset, offset + size]
            }
            offset += size

        header_json = json.dumps(header, separators=(',', ':'))
        header_json_padded = header_json + ' ' * (8 - len(header_json) % 8) if len(header_json) % 8 else header_json
        header_size = len(header_json_padded.encode())
        total_size = 8 + header_size + offset
        return total_size, header_size, header

st = SafeTensorsSimulator()
st.add_tensor('embed.weight', [32000, 4096], 'F16')
for i in range(4):
    st.add_tensor(f'layer.{i}.q.weight', [4096, 4096], 'F16')
    st.add_tensor(f'layer.{i}.k.weight', [4096, 512], 'F16')
    st.add_tensor(f'layer.{i}.v.weight', [4096, 512], 'F16')
    st.add_tensor(f'layer.{i}.o.weight', [4096, 4096], 'F16')
    st.add_tensor(f'layer.{i}.ffn.w1.weight', [4096, 11008], 'F16')
    st.add_tensor(f'layer.{i}.ffn.w2.weight', [11008, 4096], 'F16')
st.add_tensor('output.weight', [4096, 32000], 'F16')

total_size, header_size, header = st.simulate_save()

print(f"=== SafeTensors格式 ===")
print(f"张量数量: {len(st.tensors)}")
print(f"头大小: {header_size} bytes")
print(f"总文件大小: {total_size/1024/1024:.2f} MB")

print(f"\n--- JSON Header 示例（前2个张量） ---")
for i, (name, meta) in enumerate(header.items()):
    if i >= 2:
        break
    print(f"  {name}: dtype={meta['dtype']}, shape={meta['shape']}, offsets={meta['data_offsets']}")

print(f"\nSafeTensors vs PyTorch pickle:")
print(f"  安全性: 无pickle反序列化风险（无__reduce__攻击面）")
print(f"  加载速度: mmap零拷贝，比pickle快10-100x")
print(f"  惰性加载: 可只加载需要的张量（data_offsets定位）")
print(f"  跨框架: HuggingFace标准格式，Rust/Go/JS均可读取")
print(f"  分片: 大模型可拆分为多个safetensors文件")

print(f"\n--- 大模型分片策略 ---")
shard_size = 5 * 1024 * 1024 * 1024  # 5GB per shard
n_shards = max(1, int(total_size / shard_size) + (1 if total_size % shard_size else 0))
print(f"  7B FP16模型: {total_size/1024/1024/1024:.1f} GB")
print(f"  5GB分片: {n_shards} 个文件")
print(f"  文件命名: model-00001-of-{n_shards:05d}.safetensors")
print(f"  索引文件: model.safetensors.index.json (记录每个张量所在分片)")

### ONNX格式与跨框架导出

#### 什么是ONNX？

ONNX（Open Neural Network Exchange）是跨框架的模型表示标准，使用Protobuf序列化。ONNX定义了标准算子集（opset）和计算图表示，使得模型可以在PyTorch、TensorFlow、MXNet等框架间迁移。

#### 为什么ONNX重要？

1. **推理引擎入口**：TensorRT、OpenVINO、ONNX Runtime等推理引擎都以ONNX作为输入格式
2. **硬件适配**：不同硬件厂商（NVIDIA/Intel/Qualcomm/ARM）都支持ONNX算子
3. **算子标准化**：opset版本确保算子语义一致，避免框架间差异
4. **图优化基础**：ONNX图可进行常量折叠、死代码消除、算子融合等优化

#### ONNX导出的关键概念

- **动态轴（dynamic_axes）**：指定哪些维度是动态的（如batch_size、seq_len）
- **opset版本**：不同opset版本支持不同算子，推荐使用opset 14+
- **ONNX检查**：`onnx.checker.check_model()`验证模型合法性
- **自定义算子**：如果模型使用了ONNX不支持的算子，需要注册自定义算子或使用`torch.onnx.export`的`custom_opsets`

In [ ]:
class SimpleTransformerBlock(nn.Module):
    """用于ONNX导出演示的简化Transformer块"""
    def __init__(self, dim=64, n_heads=4):
        super().__init__()
        self.attn = nn.MultiheadAttention(dim, n_heads, batch_first=True)
        self.ffn = nn.Sequential(nn.Linear(dim, dim*4), nn.GELU(), nn.Linear(dim*4, dim))
        self.ln1 = nn.LayerNorm(dim)
        self.ln2 = nn.LayerNorm(dim)

    def forward(self, x):
        h = self.ln1(x)
        h, _ = self.attn(h, h, h)
        x = x + h
        x = x + self.ffn(self.ln2(x))
        return x

model = SimpleTransformerBlock(dim=64, n_heads=4)
model.eval()

dummy_input = torch.randn(1, 16, 64)

onnx_path = 'simple_transformer.onnx'
torch.onnx.export(
    model,
    dummy_input,
    onnx_path,
    opset_version=17,
    input_names=['input'],
    output_names=['output'],
    dynamic_axes={
        'input': {0: 'batch_size', 1: 'seq_len'},
        'output': {0: 'batch_size', 1: 'seq_len'},
    },
)

import onnx
onnx_model = onnx.load(onnx_path)
onnx.checker.check_model(onnx_model)

print(f"=== ONNX导出 ===")
print(f"模型: SimpleTransformerBlock")
print(f"opset版本: 17")
print(f"文件大小: {os.path.getsize(onnx_path)/1024:.1f} KB")
print(f"输入: input [batch_size, seq_len, 64]")
print(f"输出: output [batch_size, seq_len, 64]")
print(f"动态轴: batch_size, seq_len")

print(f"\n--- ONNX计算图信息 ---")
print(f"节点数: {len(onnx_model.graph.node)}")
print(f"算子类型: {set(n.op_type for n in onnx_model.graph.node)}")

print(f"\n--- 产业级ONNX导出注意事项 ---")
print(f"1. 动态轴: 必须指定batch和seq_len为动态，否则推理时只能用固定shape")
print(f"2. opset: 推荐opset 14+，支持更多算子（如GELU、LayerNorm）")
print(f"3. 验证: onnx.checker.check_model() + onnxruntime推理结果对比")
print(f"4. 优化: onnxoptimizer常量折叠+死代码消除后再送入推理引擎")
print(f"5. 量化: onnxruntime.quantization支持训练后量化(PTQ)")

os.remove(onnx_path)

### 产业级推理引擎格式

#### TensorRT (.engine/.plan)

NVIDIA GPU推理的终极格式。TensorRT将ONNX模型编译为高度优化的引擎：
- **层融合**：Conv+BN+ReLU融合为单个kernel，减少显存访问
- **内核自动调优**：在目标GPU上实测选择最优kernel实现
- **精度校准**：INT8量化时使用校准数据集确定最优量化参数
- **动态batch**：支持优化配置文件（optimal profile）指定min/opt/max shape

编译流程：`ONNX → TensorRT Builder → .engine`

注意：`.engine`文件与GPU架构绑定，A100编译的引擎无法在4090上使用。

#### TFLite (.tflite)

Google端侧推理格式，使用FlatBuffers序列化（零拷贝、无需解析）：
- 支持INT8/INT4量化、FP16推理
- 支持Android NNAPI委托、GPU委托、Edge TPU委托
- 转换流程：`Keras/TF → TFLite Converter → .tflite`

#### Core ML (.mlpackage/.mlmodelc)

Apple端侧格式，针对Apple Silicon优化：
- 自动调度到ANE（Apple Neural Engine）、GPU、CPU
- 支持INT4/INT8量化、palette量化
- 转换流程：`PyTorch → coremltools → .mlpackage`

#### QNN Context Binary

高通AI Engine Direct (QNN)的编译格式：
- 针对Hexagon DSP/NPU优化
- 支持INT8/INT4量化推理
- 转换流程：`ONNX → qnn-converter → qnn-context-binary`

In [ ]:
print(f"=== 产业级模型格式对比 ===")
formats = [
    ('GGUF', 'llama.cpp', 'Block量化+mmap', '端侧CPU推理(Apple Silicon/x86)', 'Q2_K~Q8_0'),
    ('SafeTensors', 'HuggingFace', '安全+mmap+分片', '模型分发/加载/多LoRA', 'FP16/BF16'),
    ('ONNX', '跨框架标准', '标准算子+图优化', '推理引擎输入格式', 'FP32/FP16/INT8'),
    ('TensorRT', 'NVIDIA', '层融合+内核调优', 'NVIDIA GPU推理', 'FP16/INT8/INT4'),
    ('TFLite', 'Google', 'FlatBuffers+委托', 'Android/嵌入式端侧', 'FP32/FP16/INT8'),
    ('Core ML', 'Apple', 'ANE优化+palette', 'iOS/macOS端侧', 'FP16/INT4/INT8'),
    ('QNN Binary', 'Qualcomm', 'Hexagon NPU指令', '骁龙端侧推理', 'INT8/INT4'),
]
print(f"\n{'格式':<15} {'生态':<18} {'特点':<20} {'适用场景':<28} {'精度支持':<18}")
print("-" * 99)
for name, eco, feature, scene, precision in formats:
    print(f"{name:<15} {eco:<18} {feature:<20} {scene:<28} {precision:<18}")

print(f"\n=== 产业级格式转换工作流 ===")
workflows = [
    ('训练 → CPU端侧', 'PyTorch → ONNX → GGUF (llama.cpp convert)', '量化+mmap加载'),
    ('训练 → NVIDIA GPU', 'PyTorch → ONNX → TensorRT (trtexec)', '层融合+内核调优'),
    ('训练 → Android', 'PyTorch → ONNX → TFLite (ai-edge-litert)', 'NNAPI/GPU委托'),
    ('训练 → iOS', 'PyTorch → Core ML (coremltools)', 'ANE自动调度'),
    ('训练 → 骁龙NPU', 'PyTorch → ONNX → QNN (qnn-converter)', 'Hexagon DSP优化'),
    ('HuggingFace分发', 'PyTorch → SafeTensors (safetensors库)', '安全+mmap加载'),
    ('多LoRA服务', 'SafeTensors基座 + 多个LoRA适配器', '按需切换适配器'),
]
print(f"\n{'场景':<22} {'转换路径':<50} {'关键优势':<20}")
print("-" * 92)
for scene, path, advantage in workflows:
    print(f"{scene:<22} {path:<50} {advantage:<20}")

print(f"\n=== 格式选择决策树 ===")
print(f"1. 目标硬件是什么?")
print(f"   ├─ NVIDIA GPU → TensorRT (最优性能)")
print(f"   ├─ Apple Silicon → GGUF (CPU) / Core ML (ANE)")
print(f"   ├─ Android → TFLite (NNAPI/GPU委托)")
print(f"   ├─ 骁龙NPU → QNN Context Binary")
print(f"   └─ 通用CPU → GGUF (llama.cpp)")
print(f"2. 是否需要跨框架?")
print(f"   └─ 是 → ONNX (中间表示)")
print(f"3. 是否需要安全分发?")
print(f"   └─ 是 → SafeTensors (无Pickle风险)")
print(f"4. 是否需要多LoRA?")
print(f"   └─ 是 → SafeTensors基座 + LoRA适配器 (按需mmap加载)")

---
### 模型分片与大模型加载

70B+参数的模型无法存储在单个文件中（文件系统限制、内存限制、下载效率），需要分片存储和按需加载。

#### 为什么需要分片？

1. **文件系统限制**：FAT32最大4GB，ext4最大16TB，但单个大文件下载和校验困难
2. **内存限制**：70B FP16模型约140GB，无法一次性加载到大多数设备
3. **下载效率**：分片支持并行下载和断点续传
4. **按需加载**：推理时只加载当前需要的层，减少内存占用

#### 分片策略

| 策略 | 原理 | 优点 | 缺点 |
|------|------|------|------|
| **均匀分片** | 按参数量均匀切割为N片 | 下载均衡 | 层可能被切断 |
| **按层分片** | 每N层一个分片 | 层完整性 | 分片大小不均 |
| **按组件分片** | Attention/FFN/Embedding分开 | 按需加载 | 管理复杂 |

#### HuggingFace分片规范

```
model-00001-of-00008.safetensors  (5GB)
model-00002-of-00008.safetensors  (5GB)
...
model-00008-of-00008.safetensors  (3.2GB)
model.safetensors.index.json      (元数据索引)
```

索引文件记录每个张量所在的分片：
```json
{"metadata": {"total_size": 13476836352},
 "weight_map": {
   "model.embed_tokens.weight": "model-00001-of-00008.safetensors",
   "model.layers.0.self_attn.q_proj.weight": "model-00001-of-00008.safetensors",
   ...
 }}
```

#### 分片加载延迟模型

$$T_{\text{load}} = T_{\text{index}} + \max_{i} T_{\text{shard},i}^{\text{parallel}} + T_{\text{verify}}$$

其中$T_{\text{index}}$为索引加载时间，$T_{\text{shard},i}$为第$i$个分片的加载时间，$T_{\text{verify}}$为哈希校验时间。

In [ ]:
class ModelShardingSimulator:
    """模型分片与加载模拟器"""
    def __init__(self, n_params_b, dtype_bytes=2, shard_size_gb=5):
        self.n_params = n_params_b * 1e9
        self.dtype_bytes = dtype_bytes
        self.shard_size = shard_size_gb * 1e9

    def compute_sharding(self):
        total_size = self.n_params * self.dtype_bytes
        n_shards = max(1, (total_size + self.shard_size - 1) // self.shard_size)
        shard_sizes = [min(self.shard_size, total_size - i * self.shard_size) for i in range(n_shards)]
        return {
            'total_size_gb': total_size / 1e9,
            'n_shards': n_shards,
            'shard_sizes_gb': [s / 1e9 for s in shard_sizes],
        }

    def estimate_load_time(self, bandwidth_gbs, parallel_shards=1):
        """估算分片加载时间"""
        info = self.compute_sharding()
        total_size = info['total_size_gb'] * 1e9
        index_load_s = 0.01
        if parallel_shards >= info['n_shards']:
            shard_load_s = max(info['shard_sizes_gb']) * 1e9 / (bandwidth_gbs * 1e9)
        else:
            shard_load_s = total_size / (bandwidth_gbs * 1e9)
        verify_s = info['n_shards'] * 0.1
        total_s = index_load_s + shard_load_s + verify_s
        return {
            'index_load_s': index_load_s,
            'shard_load_s': shard_load_s,
            'verify_s': verify_s,
            'total_s': total_s,
            'parallel_shards': parallel_shards,
        }

print("=== 大模型分片策略分析 ===")
for model_name, params_b, dtype in [('7B FP16', 7, 2), ('70B FP16', 70, 2), ('70B INT4', 70, 0.5)]:
    sim = ModelShardingSimulator(params_b, dtype)
    info = sim.compute_sharding()
    print(f"\n--- {model_name} ---")
    print(f"  总大小: {info['total_size_gb']:.1f} GB")
    print(f"  分片数: {info['n_shards']} (每片5GB)")

print(f"\n=== 加载时间估算 (70B FP16, SSD=5GB/s) ===")
sim_70b = ModelShardingSimulator(70, 2)
for parallel in [1, 4, 8]:
    result = sim_70b.estimate_load_time(bandwidth_gbs=5, parallel_shards=parallel)
    print(f"  并行{parallel}片: 总加载时间={result['total_s']:.1f}s")

print(f"\n=== 产业实践要点 ===")
print(f"1. HuggingFace默认5GB分片，兼顾下载效率和文件管理")
print(f"2. mmap加载时，OS按需分页，实际只需加载使用的层")
print(f"3. 并行下载可显著加速大模型加载，但受限于磁盘带宽")
print(f"4. 哈希校验(SHA256)确保分片完整性，防止损坏模型")
print(f"5. GGUF也支持分片，但分片间需保证tensor对齐")

---
### 模型加密与知识产权保护

在端侧部署中，模型文件存储在用户设备上，存在被提取、逆向和复制的风险。模型加密是保护知识产权的关键手段。

#### 威胁模型

| 攻击方式 | 难度 | 防御手段 |
|---------|------|----------|
| **直接复制模型文件** | 低 | 文件加密 + DRM |
| **内存dump提取权重** | 中 | 运行时解密 + 内存保护 |
| **逆向推理算法** | 高 | 模型混淆 + 算法白盒化 |
| **侧信道攻击** | 很高 | 常数时间实现 + 功耗随机化 |

#### 加密方案对比

| 方案 | 原理 | 安全性 | 性能开销 | 适用场景 |
|------|------|--------|---------|----------|
| **AES-256加密权重** | 权重文件加密，推理前解密 | 中 | 加载时+10-30% | 通用保护 |
| **运行时解密** | 仅解密当前使用的层 | 中高 | 推理时+5-15% | 内存受限设备 |
| **白盒加密** | 密钥嵌入推理代码，混淆密钥 | 低中 | 推理时+20-50% | 防逆向工程 |
| **平台DRM** | 利用硬件安全模块 | 高 | 极低 | iOS/Android |
| **模型混淆** | 权重重排+冗余计算 | 低 | 推理时+10-30% | 辅助手段 |

#### 平台DRM方案

- **iOS**: Keychain + Data Protection API + Secure Enclave
  - 模型密钥存储在Secure Enclave中，App无法直接读取
  - Core ML模型可绑定设备，防止跨设备复制
- **Android**: Android Keystore + Hardware-backed Keymaster
  - 密钥存储在TEE(Trusted Execution Environment)中
  - SafetyNet/Play Integrity验证设备完整性

#### 安全与性能的权衡

$$\text{Security Score} = f(\text{encryption}, \text{key management}, \text{runtime protection})$$
$$\text{Performance Overhead} = g(\text{decryption latency}, \text{memory overhead})$$

产业实践：通常选择AES-256加密+平台DRM的组合，在安全性和性能间取得平衡。

In [ ]:
class ModelEncryptionSimulator:
    """模型加密方案模拟器"""
    def __init__(self, model_size_mb):
        self.model_size_mb = model_size_mb

    def estimate_encryption_overhead(self, scheme):
        """估算加密方案的性能开销"""
        schemes = {
            'AES-256全量加密': {
                'load_overhead_pct': 15,
                'inference_overhead_pct': 0,
                'memory_overhead_pct': 5,
                'security_level': '中',
                'key_management': 'App内或Keystore',
            },
            '运行时逐层解密': {
                'load_overhead_pct': 5,
                'inference_overhead_pct': 10,
                'memory_overhead_pct': 2,
                'security_level': '中高',
                'key_management': 'Secure Enclave/TEE',
            },
            '平台DRM(iOS)': {
                'load_overhead_pct': 2,
                'inference_overhead_pct': 0,
                'memory_overhead_pct': 1,
                'security_level': '高',
                'key_management': 'Secure Enclave',
            },
            '平台DRM(Android)': {
                'load_overhead_pct': 3,
                'inference_overhead_pct': 0,
                'memory_overhead_pct': 1,
                'security_level': '高',
                'key_management': 'TEE Keymaster',
            },
        }
        return schemes.get(scheme, {})

sim = ModelEncryptionSimulator(model_size_mb=4000)
print("=== 模型加密方案对比 (7B INT4模型, ~4GB) ===")
print(f"\n{'方案':<22} {'加载开销':<10} {'推理开销':<10} {'内存开销':<10} {'安全等级':<8} {'密钥管理'}")
print("-" * 80)
for scheme in ['AES-256全量加密', '运行时逐层解密', '平台DRM(iOS)', '平台DRM(Android)']:
    info = sim.estimate_encryption_overhead(scheme)
    print(f"{scheme:<22} {info['load_overhead_pct']}%{'':<7} {info['inference_overhead_pct']}%{'':<7} "
          f"{info['memory_overhead_pct']}%{'':<7} {info['security_level']:<8} {info['key_management']}")

print(f"\n=== 产业实践要点 ===")
print(f"1. 平台DRM是最佳方案: 硬件级密钥保护，性能开销极低")
print(f"2. AES-256全量加密最通用: 跨平台兼容，但密钥管理是弱点")
print(f"3. 运行时逐层解密适合大模型: 不需要一次性解密全部权重")
print(f"4. 没有绝对安全: 确定性的攻击者总能提取模型，加密提高攻击成本")
print(f"5. 模型混淆是辅助手段: 权重重排+冗余计算增加逆向难度")
print(f"6. 法律保护同样重要: 许可协议+水印+使用监控")

---
### 模型注册中心与版本管理

产业级部署需要管理多个模型版本、支持A/B测试和灰度发布。模型注册中心（Model Registry）是基础设施。

#### 为什么需要模型注册中心？

1. **版本追踪**：每次训练产出带版本号的模型，可追溯任何线上模型对应的训练配置
2. **A/B测试**：新模型先小流量验证效果，逐步放量
3. **回滚能力**：新模型出问题时秒级回滚到上一版本
4. **血缘追踪**：记录模型→数据集→训练配置的完整链路

#### 主流模型注册中心

| 平台 | 特点 | 适用场景 |
|------|------|----------|
| **HuggingFace Hub** | 社区生态，Git-like版本管理 | 开源模型分发 |
| **MLflow Model Registry** | 实验追踪+模型注册+部署 | ML全生命周期管理 |
| **Weights & Biases** | 实验追踪+模型版本+协作 | 团队协作场景 |
| **自建Registry** | S3/MinIO+元数据DB | 企业私有化部署 |

#### 版本管理策略

- **语义版本号**：`v{major}.{minor}.{patch}`，major=架构变更，minor=重训练，patch=微调
- **哈希校验**：每个模型文件附带SHA256哈希，确保完整性
- **元数据嵌入**：训练配置、数据集版本、评估指标等嵌入模型元数据

#### 灰度发布流程

```
新模型v2.0
  │
  ▼ 1%流量
监控指标(PPL/延迟/崩溃率) → 异常? → 回滚v1.9
  │ 正常
  ▼ 10%流量
监控指标 → 异常? → 回滚
  │ 正常
  ▼ 50% → 100%流量
全量发布v2.0
```

In [ ]:
class ModelRegistry:
    """模型注册中心模拟"""
    def __init__(self):
        self.models = {}
        self.deployments = {}

    def register_model(self, name, version, model_hash, size_mb, metrics, config):
        """注册模型版本"""
        key = f"{name}:{version}"
        self.models[key] = {
            'name': name, 'version': version, 'hash': model_hash,
            'size_mb': size_mb, 'metrics': metrics, 'config': config,
            'status': 'registered',
        }

    def deploy_canary(self, name, version, traffic_pct=1):
        """灰度发布"""
        key = f"{name}:{version}"
        if key not in self.models:
            return f"模型{key}未注册"
        self.models[key]['status'] = f'canary({traffic_pct}%)'
        self.deployments[key] = {'traffic_pct': traffic_pct}
        return f"灰度发布{key}，流量{traffic_pct}%"

    def promote(self, name, version, traffic_pct=100):
        """提升流量"""
        key = f"{name}:{version}"
        self.models[key]['status'] = f'production({traffic_pct}%)'
        self.deployments[key] = {'traffic_pct': traffic_pct}
        return f"提升{key}至{traffic_pct}%流量"

    def rollback(self, name, to_version):
        """回滚"""
        for key in list(self.deployments.keys()):
            if key.startswith(f"{name}:") and to_version not in key:
                self.models[key]['status'] = 'rolled_back'
                del self.deployments[key]
        to_key = f"{name}:{to_version}"
        self.models[to_key]['status'] = 'production(100%)'
        self.deployments[to_key] = {'traffic_pct': 100}
        return f"回滚至{to_key}"

registry = ModelRegistry()
registry.register_model('qwen-1.5b', 'v1.0', 'sha256:abc123', 1200,
                        {'ppl': 5.8, 'latency_ms': 25}, {'lr': 2e-5, 'epochs': 3})
registry.register_model('qwen-1.5b', 'v1.1', 'sha256:def456', 1200,
                        {'ppl': 5.5, 'latency_ms': 25}, {'lr': 1e-5, 'epochs': 5})
registry.register_model('qwen-1.5b', 'v2.0', 'sha256:ghi789', 1350,
                        {'ppl': 5.2, 'latency_ms': 28}, {'lr': 3e-5, 'epochs': 3, 'new_data': True})

print("=== 模型注册中心 ===")
for key, info in registry.models.items():
    print(f"\n{key}:")
    print(f"  哈希: {info['hash']}")
    print(f"  大小: {info['size_mb']}MB")
    print(f"  指标: PPL={info['metrics']['ppl']}, 延迟={info['metrics']['latency_ms']}ms")
    print(f"  状态: {info['status']}")

print(f"\n=== 灰度发布流程 ===")
print(registry.deploy_canary('qwen-1.5b', 'v2.0', traffic_pct=1))
print(registry.promote('qwen-1.5b', 'v2.0', traffic_pct=10))
print(registry.promote('qwen-1.5b', 'v2.0', traffic_pct=50))
print(registry.promote('qwen-1.5b', 'v2.0', traffic_pct=100))

print(f"\n=== 回滚演示 ===")
print(registry.rollback('qwen-1.5b', 'v1.1'))
for key, info in registry.models.items():
    print(f"  {key}: {info['status']}")

print(f"\n=== 产业实践要点 ===")
print(f"1. 每个模型版本必须带哈希校验，确保部署的模型未被篡改")
print(f"2. 灰度发布: 1%→10%→50%→100%，每阶段监控关键指标")
print(f"3. 自动回滚: PPL/延迟/崩溃率超过阈值时自动回滚")
print(f"4. 血缘追踪: 记录模型→数据集→训练配置→评估结果的完整链路")
print(f"5. 多端部署: 同一模型版本可能需要为不同平台编译不同格式")

---
### 模型格式转换工具链

产业级部署中，模型需要经历多次格式转换。理解转换工具链和常见陷阱是端侧部署工程师的必备技能。

#### 转换工具链全景

```
                    ┌──→ GGUF (convert-hf-to-gguf.py)
                    ├──→ ONNX (torch.onnx.export)
PyTorch (.pt) ─────┼──→ SafeTensors (safetensors.torch)
                    ├──→ Core ML (coremltools.convert)
                    └──→ TFLite (torch→ONNX→tflite)
                                        │
ONNX ────────┬──→ TensorRT (trtexec)
             ├──→ OpenVINO (mo)
             ├──→ QNN Binary (qnn-onnx-converter)
             └──→ ONNX Runtime (直接加载)
```

#### 常见转换陷阱

| 陷阱 | 症状 | 解决方案 |
|------|------|----------|
| **算子不支持** | 转换报错 | 分解为支持的基础算子 |
| **精度漂移** | PPL异常升高 | 逐层对比余弦相似度 |
| **动态shape丢失** | 推理shape固定 | 检查dynamic_axes设置 |
| **量化格式不兼容** | 跨框架量化结果不同 | 使用目标框架自带量化 |
| **opset版本不匹配** | 新算子缺失 | 升级opset或注册自定义算子 |

#### 精度验证流程

1. **逐层余弦相似度**：$\cos(\mathbf{a}, \mathbf{b}) = \frac{\mathbf{a} \cdot \mathbf{b}}{\|\mathbf{a}\| \|\mathbf{b}\|} > 0.999$
2. **端到端PPL对比**：转换前后PPL增加应<0.5
3. **下游任务评估**：在标准benchmark上验证任务指标
4. **边界case测试**：极端输入（空字符串、超长序列）的行为一致性

In [ ]:
class ModelConversionPipeline:
    """模型格式转换管线模拟"""
    def __init__(self):
        self.conversion_graph = {
            'pytorch': ['onnx', 'safetensors', 'gguf', 'coreml', 'tflite'],
            'onnx': ['tensorrt', 'openvino', 'qnn', 'onnxruntime'],
            'safetensors': ['pytorch', 'gguf'],
        }
        self.pitfalls = {
            ('pytorch', 'onnx'): '动态shape、自定义算子、opset版本',
            ('pytorch', 'gguf'): '量化格式选择、词表对齐',
            ('onnx', 'tensorrt'): '算子兼容性、动态shape、精度校准',
            ('onnx', 'qnn'): '算子分解、NPU精度约束',
            ('pytorch', 'coreml'): 'ANE算子兼容性、State API',
        }

    def find_conversion_path(self, source, target):
        """查找转换路径"""
        if source == target:
            return [source]
        if target in self.conversion_graph.get(source, []):
            return [source, target]
        for intermediate in self.conversion_graph.get(source, []):
            path = self.find_conversion_path(intermediate, target)
            if path:
                return [source] + path
        return None

    def estimate_conversion_risk(self, source, target):
        """估算转换风险"""
        path = self.find_conversion_path(source, target)
        if not path:
            return {'path': None, 'risk': 'high', 'reason': '无直接转换路径'}
        risk_score = 0
        reasons = []
        for i in range(len(path) - 1):
            key = (path[i], path[i+1])
            if key in self.pitfalls:
                risk_score += 1
                reasons.append(f"{path[i]}→{path[i+1]}: {self.pitfalls[key]}")
        risk = 'low' if risk_score == 0 else 'medium' if risk_score <= 1 else 'high'
        return {'path': path, 'risk': risk, 'reasons': reasons}

pipeline = ModelConversionPipeline()
print("=== 模型格式转换路径分析 ===")

scenarios = [
    ('pytorch', 'gguf', '端侧CPU推理'),
    ('pytorch', 'tensorrt', 'NVIDIA GPU推理'),
    ('pytorch', 'qnn', '骁龙NPU推理'),
    ('pytorch', 'coreml', 'iOS/macOS推理'),
    ('safetensors', 'tensorrt', 'HuggingFace→GPU'),
]

for source, target, scene in scenarios:
    result = pipeline.estimate_conversion_risk(source, target)
    path_str = ' → '.join(result['path']) if result['path'] else '无路径'
    print(f"\n{scene}: {source} → {target}")
    print(f"  路径: {path_str}")
    print(f"  风险: {result['risk']}")
    for reason in result['reasons']:
        print(f"  ⚠ {reason}")

print(f"\n=== 产业级转换最佳实践 ===")
print(f"1. 最短路径优先: 每次转换都可能引入精度损失")
print(f"2. 逐层验证: 转换后必须逐层对比余弦相似度")
print(f"3. 自动化管线: 将转换+验证+打包集成到CI/CD")
print(f"4. 版本锁定: 记录每个转换工具的版本，确保可复现")
print(f"5. 回归测试: 每次框架升级后重新验证转换精度")
print(f"6. 边界case: 测试空输入、超长序列、特殊token等边界情况")

---
### 总结

#### 格式选择决策树

```
目标硬件?
├── NVIDIA GPU → TensorRT (最优性能)
├── Apple Silicon → GGUF (CPU) / Core ML (ANE)
├── Android → TFLite / QNN Binary
├── Intel CPU/NPU → OpenVINO IR
└── 通用CPU → GGUF (llama.cpp)

分发方式?
├── HuggingFace → SafeTensors (安全+mmap)
├── App内嵌 → 加密+平台DRM
└── 私有部署 → 自建Registry+S3
```

#### 产业级最佳实践

1. **SafeTensors是分发标准**：安全、快速、支持分片和按需加载
2. **GGUF是端侧CPU标准**：mmap零拷贝+K-Quant量化，llama.cpp生态
3. **ONNX是转换枢纽**：几乎所有推理引擎都接受ONNX输入
4. **加密是必须**：端侧模型必须加密，平台DRM是最佳方案
5. **版本管理是基础设施**：模型注册中心+灰度发布+自动回滚
6. **转换管线自动化**：CI/CD集成转换+验证+打包，减少人为错误